# Safebooru 메타데이터 크롤링 (Selenium 우회 버전)
Safebooru 서버 차단을 피하기 위해 Chrome 웹 크롤링(Selenium)을 사용하여 데이터를 추출합니다.
- 이미지는 저장하지 않음 — 학습 시 배치 단위로 임시 다운로드
- 저장 컬럼: id, tags, file_url, sample_url, width, height
- 기존 데이터 파일 이어서 크롤링 가능

In [1]:
import os
import time
import queue
import threading
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.chrome.options import Options

# --- 사용자 설정 ---
START_ID = 6654626     # 시작할 최상단 ID
END_ID = 6654526       # 종료할 ID
NUM_THREADS = 4        # 동시 실행할 브라우저 창 개수
SAVE_INTERVAL = 100    # 중간 저장 단위 (100개마다)
SAVE_PATH = r"C:\Users\EL069\Project\safebooru\data\metadata_html.parquet"

# --- 전역 변수 및 쓰레드 락 ---
buffer = []
save_lock = threading.Lock()
driver_pool = queue.Queue()

def create_driver():
    """웹 드라이버를 생성하여 반환합니다."""
    chrome_options = Options()
    chrome_options.add_argument('--headless=new') 
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    
    # 이미지 로딩을 차단하여 속도를 높이는 최적화 옵션
    prefs = {"profile.managed_default_content_settings.images": 2}
    chrome_options.add_experimental_option("prefs", prefs)
    
    return webdriver.Chrome(options=chrome_options)

def save_buffer():
    """버퍼에 쌓인 데이터를 파케이 파일로 병합하여 저장합니다."""
    global buffer
    if not buffer:
        return
    
    new_df = pd.DataFrame(buffer)
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    
    if os.path.exists(SAVE_PATH):
        try:
            existing_df = pd.read_parquet(SAVE_PATH)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            combined_df = combined_df.drop_duplicates(subset=['id'])
        except Exception as e:
            tqdm.write(f"파케이 파일 읽기 오류 (덮어씁니다): {e}")
            combined_df = new_df
    else:
        combined_df = new_df

    combined_df.to_parquet(SAVE_PATH, index=False)
    
    tqdm.write(f"[*] 중간 저장 완료: 누적 {len(combined_df)}건")
    
    buffer.clear()

def fetch_post_data(post_id):
    """단일 게시물의 데이터를 크롤링합니다."""
    driver = driver_pool.get()
    result_data = None
    
    try:
        target_url = f"https://safebooru.org/index.php?page=post&s=view&id={post_id}"
        driver.get(target_url)
        time.sleep(1.0) 
        
        try:
            img_element = driver.find_element(By.ID, "image")
        except NoSuchElementException:
            return None 
            
        tags = img_element.get_attribute("alt").strip()
        
        # sample_url 추출 로직 삭제됨
        
        original_link_element = driver.find_element(By.XPATH, "//a[contains(text(), 'Original image')]")
        file_url = original_link_element.get_attribute("href")
        
        size_element = driver.find_element(By.XPATH, "//div[@id='stats']//li[contains(text(), 'Size:')]")
        size_text = size_element.text.replace("Size:", "").strip()
        width, height = size_text.split("x")
        
        # 결과에서 sample_url 제외
        result_data = {
            "id": post_id,
            "tags": tags,
            "file_url": file_url,
            "width": int(width),
            "height": int(height)
        }
        
    except Exception as e:
        tqdm.write(f"[에러] ID {post_id} 파싱 실패: {e}")
    finally:
        driver_pool.put(driver)
        
    return result_data

if __name__ == "__main__":
    print(f"=== 크롤링 시작 ===")
    print(f"저장 경로: {SAVE_PATH}")
    
    for _ in range(NUM_THREADS):
        driver_pool.put(create_driver())

    total_tasks = START_ID - END_ID + 1

    with tqdm(total=total_tasks, desc="수집 진행률", ncols=100) as pbar:
        with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
            futures = {executor.submit(fetch_post_data, pid): pid for pid in range(START_ID, END_ID - 1, -1)}
            
            for future in as_completed(futures):
                data = future.result()
                if data:
                    with save_lock:
                        buffer.append(data)
                        if len(buffer) >= SAVE_INTERVAL:
                            save_buffer()
                
                pbar.update(1)

    with save_lock:
        if buffer:
            save_buffer()

    while not driver_pool.empty():
        driver = driver_pool.get()
        driver.quit()

    print("\n=== 모든 크롤링 작업이 완료되었습니다. ===")

=== 크롤링 시작 ===
저장 경로: C:\Users\EL069\Project\safebooru\data\metadata_html.parquet


수집 진행률:  99%|███████████████████████████████████████████████▌| 100/101 [00:37<00:00,  3.71it/s]     

[*] 중간 저장 완료: 누적 100건


수집 진행률: 100%|████████████████████████████████████████████████| 101/101 [00:38<00:00,  2.61it/s]


[*] 중간 저장 완료: 누적 101건

=== 모든 크롤링 작업이 완료되었습니다. ===
